# Notebook 06 Combine External Judges dan Report Gabungan

Notebook ini dijalankan setelah notebook 05 selesai menghasilkan `evaluation_table.csv`. Fungsinya menggabungkan evaluasi internal dengan file external judge GPT5.5 dan Gemini3.5, lalu membuat tabel, workbook, report markdown, dan plot gabungan.


In [ ]:
import re
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("../data/eval_outputs")
PLOT_DIR = OUTPUT_DIR / "plots"
EVALUATION_TABLE_PATH = OUTPUT_DIR / "evaluation_table.csv"
EXTERNAL_GPT_PATH = OUTPUT_DIR / "external_llm_as_a_judge_GPT5.5.xlsx"
EXTERNAL_GEMINI_PATH = OUTPUT_DIR / "external_llm_as_a_judge_Gemini3.5.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("eval table", EVALUATION_TABLE_PATH.resolve())
print("gpt judge", EXTERNAL_GPT_PATH.resolve())
print("gemini judge", EXTERNAL_GEMINI_PATH.resolve())


In [ ]:
def normalize_id(value) -> str:
    text = str(value).strip().upper()
    match = re.search(r"(\d+)", text)
    if not match:
        return text
    return f"Q{int(match.group(1)):02d}"


def require_file(path: Path, instruction: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}. {instruction}")


def load_external_judges() -> tuple[pd.DataFrame, pd.DataFrame]:
    require_file(EXTERNAL_GPT_PATH, "Letakkan workbook GPT5.5 external judge di data/eval_outputs.")
    require_file(EXTERNAL_GEMINI_PATH, "Letakkan workbook Gemini3.5 external judge di data/eval_outputs.")

    gpt = pd.read_excel(EXTERNAL_GPT_PATH)
    gpt = gpt.rename(columns={
        "ID": "id",
        "Nomor Pertanyaan": "id",
        "Nilai": "external_judge_gpt55_score_raw",
        "Alasan ketat": "external_judge_gpt55_reason",
        "Alasan Kuat": "external_judge_gpt55_reason",
    })
    gpt["id"] = gpt["id"].map(normalize_id)
    gpt["external_judge_gpt55_score"] = pd.to_numeric(gpt["external_judge_gpt55_score_raw"], errors="coerce") / 10.0

    gemini = pd.read_excel(EXTERNAL_GEMINI_PATH)
    gemini = gemini.rename(columns={
        "ID": "id",
        "Nomor Pertanyaan": "id",
        "Nilai": "external_judge_gemini35_score_raw",
        "Alasan ketat": "external_judge_gemini35_reason",
        "Alasan Kuat": "external_judge_gemini35_reason",
    })
    gemini["id"] = gemini["id"].map(normalize_id)
    gemini["external_judge_gemini35_score"] = pd.to_numeric(gemini["external_judge_gemini35_score_raw"], errors="coerce") / 10.0

    return gpt, gemini


require_file(EVALUATION_TABLE_PATH, "Jalankan notebook 05 terlebih dahulu untuk membuat evaluation_table.csv.")
df = pd.read_csv(EVALUATION_TABLE_PATH)
df["id"] = df["id"].map(normalize_id)

gpt_judge, gemini_judge = load_external_judges()
combined = df.merge(
    gpt_judge[["id", "external_judge_gpt55_score_raw", "external_judge_gpt55_score", "external_judge_gpt55_reason"]],
    on="id",
    how="left",
).merge(
    gemini_judge[["id", "external_judge_gemini35_score_raw", "external_judge_gemini35_score", "external_judge_gemini35_reason"]],
    on="id",
    how="left",
)

external_cols = ["external_judge_gpt55_score", "external_judge_gemini35_score"]
combined["external_judge_mean_score"] = combined[external_cols].mean(axis=1)
combined["external_judge_min_score"] = combined[external_cols].min(axis=1)
combined["external_judge_max_score"] = combined[external_cols].max(axis=1)
combined["external_judge_disagreement"] = (combined["external_judge_max_score"] - combined["external_judge_min_score"]).fillna(0.0)
combined["internal_external_gap"] = combined["overall_score"] - combined["external_judge_mean_score"]
combined["overall_score_with_external"] = (combined["overall_score"] * 0.50) + (combined["external_judge_mean_score"] * 0.50)
combined["external_quality_label"] = pd.cut(combined["external_judge_mean_score"], bins=[-0.01, 0.50, 0.70, 0.85, 1.01], labels=["poor", "fair", "good", "excellent"])
combined["combined_quality_label"] = pd.cut(combined["overall_score_with_external"], bins=[-0.01, 0.50, 0.70, 0.85, 1.01], labels=["poor", "fair", "good", "excellent"])

missing_external = combined[combined[external_cols].isna().any(axis=1)]["id"].tolist()
if missing_external:
    print("warning: external judge belum lengkap untuk", missing_external)

combined[["id", "overall_score", "external_judge_mean_score", "overall_score_with_external", "combined_quality_label"]].head()


In [ ]:
summary_metrics = [
    "overall_score", "llm_judge_score", "semantic_similarity", "keyword_coverage",
    "retrieval_citation_coverage", "retrieval_law_hit", "retrieval_article_hit",
    "answer_citation_hit", "external_judge_gpt55_score", "external_judge_gemini35_score",
    "external_judge_mean_score", "external_judge_disagreement", "internal_external_gap",
    "overall_score_with_external",
]
summary_metrics = [c for c in summary_metrics if c in combined.columns]

summary = combined[summary_metrics].agg(["mean", "median", "min", "max", "std"]).T.reset_index().rename(columns={"index": "metric"})
by_topic = combined.groupby("topic", dropna=False).agg(
    n=("id", "count"),
    overall_score=("overall_score", "mean"),
    external_judge_mean_score=("external_judge_mean_score", "mean"),
    overall_score_with_external=("overall_score_with_external", "mean"),
    external_judge_disagreement=("external_judge_disagreement", "mean"),
    keyword_coverage=("keyword_coverage", "mean"),
    retrieval_citation_coverage=("retrieval_citation_coverage", "mean"),
).reset_index()

external_judge_comparison = combined[[
    "id", "topic", "question", "overall_score", "llm_judge_score",
    "external_judge_gpt55_score_raw", "external_judge_gpt55_score",
    "external_judge_gemini35_score_raw", "external_judge_gemini35_score",
    "external_judge_mean_score", "external_judge_disagreement", "internal_external_gap",
    "overall_score_with_external", "quality_label", "external_quality_label", "combined_quality_label",
    "external_judge_gpt55_reason", "external_judge_gemini35_reason",
]].sort_values("overall_score_with_external")

failure_cols = [
    "id", "topic", "question", "expected_answer", "answer",
    "overall_score", "external_judge_mean_score", "overall_score_with_external",
    "external_judge_disagreement", "internal_external_gap", "combined_quality_label",
    "keyword_missing", "expected_article_rank", "top1_reference",
    "external_judge_gpt55_reason", "external_judge_gemini35_reason",
]
failure_cols = [c for c in failure_cols if c in combined.columns]
external_failure_review = combined.sort_values("overall_score_with_external")[failure_cols]

combined_csv = OUTPUT_DIR / "evaluation_table_with_external_judges.csv"
external_summary_csv = OUTPUT_DIR / "evaluation_summary_with_external_judges.csv"
external_by_topic_csv = OUTPUT_DIR / "evaluation_by_topic_with_external_judges.csv"
external_judge_csv = OUTPUT_DIR / "external_judge_comparison.csv"
external_excel_path = OUTPUT_DIR / "evaluation_report_with_external_judges.xlsx"

combined.to_csv(combined_csv, index=False, encoding="utf-8-sig")
summary.to_csv(external_summary_csv, index=False, encoding="utf-8-sig")
by_topic.to_csv(external_by_topic_csv, index=False, encoding="utf-8-sig")
external_judge_comparison.to_csv(external_judge_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(external_excel_path, engine="openpyxl") as writer:
    combined.to_excel(writer, index=False, sheet_name="detail")
    summary.to_excel(writer, index=False, sheet_name="summary")
    by_topic.to_excel(writer, index=False, sheet_name="by_topic")
    external_judge_comparison.to_excel(writer, index=False, sheet_name="external_judges")
    external_failure_review.to_excel(writer, index=False, sheet_name="failure_review")

for path in [combined_csv, external_summary_csv, external_by_topic_csv, external_judge_csv, external_excel_path]:
    print("saved", path.resolve())


In [ ]:
def save_fig(name: str):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path.resolve())
    plt.show()
    plt.close()

x = np.arange(len(combined))
labels = combined["id"].astype(str).tolist()

plt.figure(figsize=(13, 6))
plt.plot(x, combined["overall_score"], marker="o", label="internal overall")
plt.plot(x, combined["external_judge_mean_score"], marker="o", label="external mean")
plt.plot(x, combined["overall_score_with_external"], marker="o", label="combined")
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("Internal, External, and Combined Score per Question")
plt.xlabel("Question ID")
plt.ylabel("Score")
plt.legend()
save_fig("13_internal_external_combined_scores.png")

plt.figure(figsize=(13, 6))
width = 0.35
plt.bar(x - width / 2, combined["external_judge_gpt55_score"], width, label="GPT5.5")
plt.bar(x + width / 2, combined["external_judge_gemini35_score"], width, label="Gemini3.5")
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("External Judge Scores per Question")
plt.xlabel("Question ID")
plt.ylabel("Normalized score")
plt.legend()
save_fig("14_external_judge_scores.png")

plt.figure(figsize=(13, 5))
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in combined["internal_external_gap"].fillna(0)]
plt.bar(labels, combined["internal_external_gap"], color=colors)
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.title("Internal minus External Score Gap")
plt.xlabel("Question ID")
plt.ylabel("Gap")
save_fig("15_internal_external_gap.png")

plt.figure(figsize=(13, 5))
plt.bar(labels, combined["external_judge_disagreement"])
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("External Judge Disagreement")
plt.xlabel("Question ID")
plt.ylabel("Score range")
save_fig("16_external_judge_disagreement.png")

plot_topic = by_topic.sort_values("overall_score_with_external")
plt.figure(figsize=(10, 5))
plt.barh(plot_topic["topic"].astype(str), plot_topic["overall_score_with_external"])
plt.xlim(0, 1)
plt.title("Combined Score by Topic")
plt.xlabel("Mean combined score")
save_fig("17_combined_score_by_topic.png")

plt.figure(figsize=(8, 5))
plt.scatter(combined["overall_score"], combined["external_judge_mean_score"])
for _, row in combined.iterrows():
    plt.text(row["overall_score"], row["external_judge_mean_score"], str(row["id"]), fontsize=8)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.title("Internal versus External Mean Score")
plt.xlabel("Internal overall score")
plt.ylabel("External mean score")
save_fig("18_internal_vs_external_scatter.png")


In [ ]:
plot_descriptions = [
    ("01_overall_score_per_question.png", "Skor akhir internal per pertanyaan."),
    ("02_overall_score_distribution.png", "Distribusi skor internal."),
    ("03_average_metric_scores.png", "Rata-rata tiap metrik internal."),
    ("04_score_by_topic.png", "Rata-rata skor internal per topik hukum."),
    ("05_latency_vs_score.png", "Hubungan latency dengan skor."),
    ("06_metric_heatmap.png", "Heatmap metrik internal per pertanyaan."),
    ("07_quality_label_count.png", "Jumlah label kualitas internal."),
    ("08_expected_article_rank.png", "Rank pasal target dalam hasil retrieval."),
    ("09_semantic_vs_keyword.png", "Semantic similarity versus keyword coverage."),
    ("10_score_spread_by_topic.png", "Sebaran skor per topik."),
    ("11_answer_word_count.png", "Panjang jawaban per pertanyaan."),
    ("12_binary_hit_stack.png", "Stack hit retrieval/citation biner."),
    ("13_internal_external_combined_scores.png", "Perbandingan skor internal, external mean, dan gabungan."),
    ("14_external_judge_scores.png", "Perbandingan skor GPT5.5 dan Gemini3.5."),
    ("15_internal_external_gap.png", "Selisih skor internal dan external."),
    ("16_external_judge_disagreement.png", "Tingkat beda penilaian antar external judge."),
    ("17_combined_score_by_topic.png", "Skor gabungan per topik."),
    ("18_internal_vs_external_scatter.png", "Scatter skor internal versus external."),
]

plot_lines = "\n".join([f"- `plots/{name}`: {desc}" for name, desc in plot_descriptions])
worst_external = external_failure_review.head(5)
best_external = combined.sort_values("overall_score_with_external", ascending=False).head(5)
model_name = combined["model_id"].mode().iloc[0] if "model_id" in combined and not combined["model_id"].dropna().empty else "-"

combined_report = f"""# Combined RAG Evaluation Report

Report ini menggabungkan evaluasi otomatis internal, local LLM-as-a-judge, dan external LLM-as-a-judge dari GPT5.5 serta Gemini3.5.

## Dataset

- Jumlah pertanyaan: {len(combined)}
- Model jawaban: {model_name}
- External judge files:
  - `external_llm_as_a_judge_GPT5.5.xlsx`
  - `external_llm_as_a_judge_Gemini3.5.xlsx`

## File Output Gabungan

- `evaluation_table_with_external_judges.csv`
- `evaluation_summary_with_external_judges.csv`
- `evaluation_by_topic_with_external_judges.csv`
- `external_judge_comparison.csv`
- `evaluation_report_with_external_judges.xlsx`
- `EVALUATION_COMBINED_REPORT.md`

## Ringkasan Skor

| Metrik | Nilai |
|---|---:|
| Internal overall mean | {combined['overall_score'].mean():.3f} |
| Local LLM judge mean | {combined['llm_judge_score'].mean():.3f} |
| GPT5.5 judge mean | {combined['external_judge_gpt55_score'].mean():.3f} |
| Gemini3.5 judge mean | {combined['external_judge_gemini35_score'].mean():.3f} |
| External judge mean | {combined['external_judge_mean_score'].mean():.3f} |
| Combined internal+external mean | {combined['overall_score_with_external'].mean():.3f} |
| Retrieval citation coverage | {combined['retrieval_citation_coverage'].mean():.3f} |
| Keyword coverage | {combined['keyword_coverage'].mean():.3f} |
| Mean external disagreement | {combined['external_judge_disagreement'].mean():.3f} |

## Interpretasi Metrik

- `overall_score`: skor otomatis internal dari semantic similarity, keyword coverage, retrieval citation coverage, retrieval/article hit, dan metrik pendukung lain.
- `llm_judge_score`: skor local LLM-as-a-judge dari notebook 05.
- `external_judge_gpt55_score` dan `external_judge_gemini35_score`: skor external judge yang dinormalisasi dari 0-10 menjadi 0-1.
- `external_judge_mean_score`: rata-rata dua external judge.
- `external_judge_disagreement`: selisih skor tertinggi dan terendah antar external judge.
- `internal_external_gap`: `overall_score - external_judge_mean_score`.
- `overall_score_with_external`: rata-rata 50% `overall_score` dan 50% `external_judge_mean_score`.

## Lima Skor Gabungan Terendah

{worst_external[['id', 'topic', 'overall_score', 'external_judge_mean_score', 'overall_score_with_external', 'combined_quality_label']].to_markdown(index=False)}

## Lima Skor Gabungan Tertinggi

{best_external[['id', 'topic', 'overall_score', 'external_judge_mean_score', 'overall_score_with_external', 'combined_quality_label']].to_markdown(index=False)}

## Plot Evaluasi

{plot_lines}

## Catatan Penggunaan

Untuk laporan akhir, gunakan `overall_score_with_external` sebagai skor gabungan karena sudah menggabungkan sinyal otomatis/retrieval dengan penilaian external judge. Gunakan `external_judge_comparison.csv` untuk melihat alasan ketat dari masing-masing external judge, terutama pada pertanyaan dengan `external_judge_disagreement` tinggi atau `internal_external_gap` besar.
"""

combined_report_path = OUTPUT_DIR / "EVALUATION_COMBINED_REPORT.md"
combined_report_path.write_text(combined_report, encoding="utf-8")
print(combined_report)
print("saved", combined_report_path.resolve())
